In [ ]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_short.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

category_map = pd.read_csv(
    "../data/reference/category_map.csv",
    sep=";",
    index_col=0,
)

In [ ]:
category_map['category'].value_counts()

1

In [ ]:
period_bins = [1800, 1920, 1940, 1960, 1990, 2030]
period_labels = ["before 1920", "1920-1939", "1940-1959", "1960-1989", "1990+"]

data_styles_periods = (
    data_gdynia.assign(
        periods=lambda d: pd.cut(
            d["build_year_est"], period_bins, right=False, labels=period_labels
        )
    )
    .groupby(["arch_style", "periods"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
assert (data_styles_periods.sum(axis=0) == [2, 246, 267, 353, 114]).all()

In [ ]:
data_styles_periods

2.

In [ ]:
storey_bins = [1, 2, 5, 9, 50]
storey_labels = ["1", "2-4", "5-8", "9+"]

data_storeys = data_gdynia.assign(
    cond_num=lambda d: pd.cut(
        d["number_of_storeys"], storey_bins, right=False, labels=storey_labels
    )
)


In [ ]:
assert (data_storeys.groupby("cond_num").size() == [379, 132, 430, 41]).all()

In [ ]:
category_map["function_clean"].duplicated(keep=False).sum()

3.

In [ ]:
data_gdynia_upper_categories = (
    data_gdynia.join(category_map, on="upper_floor_function", validate="many_to_one")
    .drop(columns=["function_clean", "subcategory"])
    .rename(columns={"category": "upper_floor_category"})
)
is_type = (
    data_gdynia_long_categories.assign(
        is_garage=lambda d: d["category"] == "garaże",
        is_residential=lambda d: d["category"] == "mieszkaniowa",
        is_vacant=lambda d: d["category"] == "pustostan",
        is_other_service=lambda d: (
            ~d["category"].isin(["garaże", "mieszkaniowa", "pustostan", "brak_parteru"])
        ),
    )
    .groupby("id")[["is_garage", "is_residential", "is_vacant", "is_other_service"]]
    .any()
)

In [ ]:
def condlist(d):
    return [
        d["number_of_storeys"].ge(10),
        d["is_garage"],
        d["is_other_service"] & (d["upper_floor_category"] == "mieszkaniowa"),
        d["is_other_service"],
        d["is_residential"] | (d["upper_floor_category"] == "mieszkaniowa"),
    ]


choicelist_type = ["high rise", "garage", "mixed-use", "service", "residential"]

building_types = (
    pd.merge(
        data_gdynia_upper_categories,
        is_type,
        on="id",
    )
    .assign(building_type=lambda d: np.select(condlist(d), choicelist_type, "other"))
    .loc[:, ["addr_street", "building_type"]]
)

In [ ]:
building_types["building_type"].value_counts()

4.

In [ ]:
data_confidence_where = data_gdynia.assign(
    build_year_est=lambda d: d["build_year_est"].where(d["confidence"] != 1, pd.NA)
)
data_confidence_mask = data_gdynia.assign(
    build_year_est=lambda d: d["build_year_est"].mask(d["confidence"] == 1, pd.NA)
)

In [ ]:
pd.testing.assert_series_equal(
    data_confidence_where["build_year_est"],
    data_confidence_mask["build_year_est"],
    check_names=False,
)

In [ ]:
data_confidence_where.groupby("arch_style")["build_year_est"].mean().sub(
    data_gdynia.groupby("arch_style")["build_year_est"].mean()
)

In [ ]:
data_confidence_where.assign(
    arch_style=lambda d: d["arch_style"].where(d["confidence"] != 1, "unknown")
)["arch_style"].value_counts()

5.

In [ ]:
dates_missing_day = (
    data_gdynia.assign(is_conf_3=lambda d: d["confidence"] == 3)
    .resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
        mean_storeys=("number_of_storeys", "mean"),
        share_conf_3=("is_conf_3", "mean"),
    )
)

In [ ]:
dates_missing_day_interpolate = dates_missing_day.assign(
    num_buildings=lambda d: d["num_buildings"].where(d["num_buildings"] != 0, pd.NA)
).interpolate()

In [ ]:
assert np.isclose(
    dates_missing_day["mean_storeys"].ffill().loc["2026-09-28"], 3.6334, atol=0.001
)
assert np.isclose(
    dates_missing_day_interpolate["mean_storeys"].loc["2026-09-28"], 4.43, atol=0.001
)

In [ ]:
dates_missing_day_interpolate

6.

In [ ]:
dates_missing_day.loc[:, "num_buildings"]

In [ ]:
dates_missing_day.loc[:, "num_buildings"].rolling(3).sum()

In [ ]:
dates_missing_day.loc[:, "num_buildings"].rolling(3, min_periods=1).sum()

In [ ]:
data_gdynia.groupby("date_of_checking").size().rolling(3).sum()

In [ ]:
data_gdynia.groupby("date_of_checking").size().rolling("3D", min_periods=1).sum()

7.

In [ ]:
days_between_street_buildings = (
    data_gdynia.assign(
        first_check_street=lambda d: d.groupby("addr_street")[
            "date_of_checking"
        ].transform("min")
    )
    .assign(
        days_since_street_first_check=lambda d: (
            (d["date_of_checking"] - d["first_check_street"]).dt.days
        )
    )
    .loc[
        :,
        [
            "addr_street",
            "date_of_checking",
            "first_check_street",
            "days_since_street_first_check",
        ],
    ]
)

In [ ]:
days_between_street_buildings[
    "days_since_street_first_check"
].value_counts().to_frame().assign(
    share_of_buildings_surveyed=lambda d: (
        d["count"] / data_gdynia["addr_street"].count()
    )
).sort_index()


In [ ]:
assert (
    len(
        days_between_street_buildings["days_since_street_first_check"].loc[
            lambda d: d.ge(5)
        ]
    )
    == 109
)

In [ ]:
days_between_street_buildings.loc[
    days_between_street_buildings["days_since_street_first_check"].ge(5)
].groupby("addr_street").size().sort_values(ascending=False)

8.

In [ ]:
data_gdynia.resample("D", on="date_of_checking").size().to_frame(
    "num_buildings"
).assign(max_record=lambda d: d["num_buildings"].cummax()).assign(
    record_yesterday=lambda d: d["max_record"].shift(1, fill_value=0)
).assign(is_record=lambda d: d["max_record"] > d["record_yesterday"])

In [ ]:
all_days = pd.date_range(
    data_gdynia["date_of_checking"].min(),
    data_gdynia["date_of_checking"].max(),
    freq="D",
)


first_visit = data_gdynia.groupby("addr_street")["date_of_checking"].min()
last_visit = data_gdynia.groupby("addr_street")["date_of_checking"].max()

streets_in_progress = pd.concat(
    {
        "num_opened": first_visit.value_counts().reindex(all_days, fill_value=0),
        "num_closed": last_visit.value_counts().reindex(all_days, fill_value=0),
    },
    axis=1,
).assign(in_progress=lambda d: d["num_opened"].cumsum() - d["num_closed"].cumsum())

9.

In [ ]:
dates = data_gdynia.groupby("date_of_checking")[["number_of_storeys"]].mean()
pd.concat(
    {
        "num_cond": dates["number_of_storeys"],
        "shift_wo_frequency": dates["number_of_storeys"].shift(1),
        "shift_with_frequency": dates["number_of_storeys"].shift(1, freq="D"),
    },
    axis=1,
    sort=True,
)

10.

In [ ]:
share_1920_1939_garages = (
    data_gdynia.assign(
        building_periods=lambda d: pd.cut(
            d["build_year_est"], period_bins, right=False, labels=period_labels
        ),
        is_1920_1939=lambda d: d["building_periods"] == "1920-1939",
    )
    .join(
        data_gdynia_long_categories.assign(
            is_garage=lambda d: d["category"] == "garaże"
        )
        .groupby("id")
        .any()
        .loc[:, "is_garage"],
        validate="1:1",
    )
    .resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
        share_1920_39=("is_1920_1939", "mean"),
        share_garages=("is_garage", "mean"),
    )
    .assign(
        share_1920_39=lambda d: d["share_1920_39"].where(
            d["num_buildings"] >= 30, pd.NA
        ),
        share_garages=lambda d: d["share_garages"].where(
            d["num_buildings"] >= 30, pd.NA
        ),
    )
)

In [ ]:
print(
    f"day with biggest share of build 1920-1939: {share_1920_1939_garages['share_1920_39'].idxmax()} with share: {share_1920_1939_garages['share_1920_39'].max()}"
)
print(
    f"day with biggest share garages: {share_1920_1939_garages['share_garages'].idxmax()} with share: {share_1920_1939_garages['share_garages'].max()}"
)

In [ ]:
share_1920_1939_garages